<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 34px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Assemble the report</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        Collect every chart from notebooks 1&ndash;3 into <strong>one self-contained HTML report</strong> and a matching <strong>data workbook</strong>, then open the report inside TIP.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; landscape analyses after <span style="color: #be0f05; font-weight: 600;">Riccardo Priore</span>, Centro PATLIB, AREA Science Park
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 680px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What this notebook does</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Collect every analysis contribution<br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Build the self-contained HTML report<br/>Step&nbsp;3 &nbsp;&middot;&nbsp; Build the data workbook (one sheet per chart)<br/>Step&nbsp;4 &nbsp;&middot;&nbsp; Open the report
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 680px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Part 4 of 4 &mdash; run the four notebooks of this module in order.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Re-running needs <strong>EPO&nbsp;TIP</strong> (it queries PATSTAT&nbsp;PROD via <code>epo.tipdata</code>).
            Each notebook writes what the next one reads; notebook&nbsp;4 assembles the report.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global.
    </div>
</div>

## Step 1 — Collect every analysis contribution

Each earlier notebook left an inline figure fragment plus its data in its output folder, noted in a small manifest. `report_kit.load_contributions` gathers them all and orders them by their `order` number, so the report reads top-to-bottom in a deliberate sequence.

In [ ]:
from pathlib import Path
import report_kit

ROOT = Path.cwd()
entries = report_kit.load_contributions(ROOT)
for e in entries:
    print(f"{e['order']:>4}  {e['title']}  ({len(e['sheets'])} sheet(s))")
print(f"\n{len(entries)} contributions.")

## Step 2 — Build the self-contained HTML report

We embed **one** copy of plotly.js in the page head, then stack every figure fragment as a section. No iframes anywhere — that is exactly what lets the report render inside TIP (Jupyter's `/files/` sandbox blocks the JavaScript an iframe would need). The result is one portable file that also works offline in any browser.

In [ ]:
from plotly.offline import get_plotlyjs

REPORT_DIR = ROOT / "report"
REPORT_DIR.mkdir(exist_ok=True)
report_path = REPORT_DIR / "antibiotic_resistance_report.html"

sections = []
for e in entries:
    fragment = Path(e["fragment_path"]).read_text(encoding="utf-8")
    note = f'<p class="note">{e["note"]}</p>' if e.get("note") else ""
    sections.append(
        f'<section><h2>{e["title"]}</h2>{note}<div class="chart">{fragment}</div></section>')

html = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Antibiotic Resistance — Patent Landscape</title>
<script>{get_plotlyjs()}</script>
<style>
  :root {{ --accent:#be0f05; --ink:#1e293b; --muted:#64748b; --line:#e2e8f0; --bg:#fff; --panel:#f8fafc; }}
  @media (prefers-color-scheme: dark) {{
    :root {{ --ink:#e2e8f0; --muted:#94a3b8; --line:#334155; --bg:#0f172a; --panel:#1e293b; }}
  }}
  * {{ box-sizing:border-box; }}
  body {{ margin:0; font-family:system-ui,-apple-system,sans-serif; color:var(--ink); background:var(--bg); }}
  header {{ padding:40px 24px 28px; text-align:center; border-bottom:1px solid var(--line); }}
  .brand {{ display:inline-block; background:var(--accent); color:#fff; padding:8px 16px;
            border-radius:10px; font-weight:800; letter-spacing:-.3px; font-size:22px; }}
  .sub {{ color:var(--muted); font-size:13px; margin-top:12px; }}
  main {{ max-width:1040px; margin:0 auto; padding:8px 24px 64px; }}
  section {{ padding:28px 0; border-bottom:1px solid var(--line); }}
  h2 {{ font-size:20px; margin:0 0 4px; }}
  .note {{ color:var(--muted); font-size:13px; margin:0 0 14px; }}
  .chart {{ width:100%; overflow-x:auto; }}
  footer {{ text-align:center; color:var(--muted); font-size:12px; padding:28px 24px; }}
</style></head>
<body>
<header>
  <div class="brand">TIP4PATLIBS — Antibiotic Resistance</div>
  <div class="sub">Patent landscape · EPO PATSTAT Global · landscape analyses after Riccardo Priore, Centro PATLIB, AREA Science Park</div>
</header>
<main>
{"".join(sections)}
</main>
<footer>Generated on EPO TIP from PATSTAT PROD · {len(entries)} analyses · self-contained, no internet needed.</footer>
</body></html>"""

report_path.write_text(html, encoding="utf-8")
print(f"Wrote {report_path}  —  {report_path.stat().st_size/1e6:.1f} MB, {len(entries)} sections.")

## Step 3 — Build the data workbook (one sheet per chart)

Every chart's underlying data goes into one Excel workbook, one sheet per chart (a network contributes two: nodes and edges). This is the "show me the numbers" companion — the report is auditable and a client can reuse the data without touching TIP.

In [ ]:
import re
import pandas as pd

data_path = REPORT_DIR / "antibiotic_resistance_report_data.xlsx"

def sheet_name(base, used):
    name = re.sub(r"[\\/*?:\[\]]", " ", base)[:31].strip() or "sheet"
    candidate, i = name, 1
    while candidate in used:
        i += 1
        candidate = f"{name[:28]}_{i}"
    used.add(candidate)
    return candidate

used = set()
with pd.ExcelWriter(data_path, engine="openpyxl") as writer:
    for e in entries:
        multi = len(e["sheets"]) > 1
        for label, path in e["sheet_paths"].items():
            # for multi-table charts keep the label (nodes/edges) visible within Excel's 31-char limit
            base = f"{e['title'][:22]} ({label})" if multi else e["title"]
            pd.read_parquet(path).to_excel(writer, sheet_name=sheet_name(base, used), index=False)

print(f"Wrote {data_path}  —  {len(used)} sheet(s).")

## Step 4 — Open the report

The report is self-contained HTML, so we open it with the course's shared `open_html` helper: it serves the file through jupyter-server-proxy and shows a red **Open** button (plus a download link). That is the reliable way to view an interactive HTML artifact inside TIP — a plain link or an iframe would be blocked by Jupyter's sandbox.

In [ ]:
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists()), Path.cwd())
sys.path.insert(0, str(root / "1_startwithtip"))
from tip_tools import open_html

open_html(report_path, "the antibiotic-resistance report")

---
**Done.** The report and its data workbook are in `report/`. This is the MVP spine: dataset → three analyses → one self-contained report that renders in TIP, plus a matching data workbook. Further analyses slot in by following the same `report_kit.record` contract.